# Laboratorio 7
Aprendizaje por Refuerzo

- Joaquín Puente
- Daniel Dubón
- Diego Valenzuela
- Nelson García Bravatti


Una empresa de ciberseguridad opera un sistema de detección de intrusiones en redes corporativas. El
sistema monitorea el tráfico de red en tiempo real y debe decidir qué acciones tomar ante comportamientos
sospechosos: ignorar, alertar, bloquear temporalmente, o aislar el segmento afectado. El estado del sistema
en cada instante es un vector de alta dimensión que incluye métricas de tráfico, patrones de comportamiento
de usuarios, y logs de eventos recientes. La empresa quiere explorar si Deep RL puede mejorar la toma de
decisiones frente a los sistemas basados en reglas que usan actualmente.


# Task 1 (Entrega Parcial)
## Task 1.1

El equipo de ingeniería propone usar DQN estándar para este sistema. Antes de implementar cualquier cosa,
usted debe hacer un análisis crítico de esa propuesta.

a. El estado del sistema es un vector de 256 dimensiones con valores continuos. Argumenten si DQN
es arquitecturalmente apropiado para este tipo de estado. ¿Qué tipo de red usarían como
aproximador de 𝑄(𝑠, 𝑎;𝒘) y por qué, considerando que el estado no es una imagen sino un vector
numérico?

b. El espacio de acciones tiene 4 opciones discretas. Argumenten si DQN o alguna de sus variantes es
apropiado para este espacio de acciones, o si sería necesario un algoritmo diferente.

c. En el dominio de detección de intrusiones, los eventos críticos (intrusiones reles) son
extremadamente raros: ocurren en menos del 0.1% de los pasos de tiempo. Argumenten
formalmente qué consecuencia tiene esa rareza sobre el buffer de Experience Replay y sobre la
distribución de muestreo uniforme. ¿Qué variante de DQN resolvería este problema específico y
cómo?

d. El sistema actual basado en reglas tiene una tasa de falsos positivos del 8%. Si diseñaran la función
de recompensa de DQN para minimizar falsos positivos únicamente, ¿qué comportamiento
indeseable podría aprender el agente? Diseñen una función de recompensa con al menos tres
componentes que capture el objetivo real del sistema, justificando la magnitud relativa de cada uno.



### 1.1.a 

**DQN sí es arquitecturalmente apropiado para un estado de 256 variables continuas.** La restricción de DQN está en el espacio de acciones, que debe ser discreto y manejable, no en que los estados sean discretos. Una tabla no permitiría representar todos los vectores posibles de $\mathbb{R}^{256}$; una red neuronal permite aproximar valores y generalizar entre estados similares.

Usaríamos un **perceptrón multicapa (MLP)** con capas completamente conectadas. Una arquitectura inicial, sujeta a validación, sería:

$$
\mathbf{s}\in\mathbb{R}^{256}
\longrightarrow \operatorname{Dense}(256)+\operatorname{ReLU}
\longrightarrow \operatorname{Dense}(128)+\operatorname{ReLU}
\longrightarrow \operatorname{Dense}(4).
$$

La salida sería el vector

$$
Q(\mathbf{s},\cdot;\mathbf{w})=
\begin{bmatrix}
Q(\mathbf{s},\text{ignorar};\mathbf{w}) &
Q(\mathbf{s},\text{alertar};\mathbf{w}) &
Q(\mathbf{s},\text{bloquear};\mathbf{w}) &
Q(\mathbf{s},\text{aislar};\mathbf{w})
\end{bmatrix}.
$$

La última capa tendría activación **lineal**, porque los valores $Q$ son retornos esperados, posiblemente negativos, y no probabilidades. No usaríamos *softmax*. Estandarizaríamos las entradas con estadísticas del conjunto de entrenamiento para que, por ejemplo, el volumen de tráfico no domine por su escala a otras métricas.

Una MLP puede aprender interacciones no lineales entre las variables. Una CNN no sería la primera elección porque el orden de las 256 métricas no implica vecindad espacial ni patrones trasladables como los de una imagen. Tampoco convertiríamos arbitrariamente el vector en una imagen.

La viabilidad depende además de la representación: el vector debe resumir suficiente historia para aproximar un estado de Markov. Si dos situaciones con el mismo vector requieren decisiones diferentes por eventos anteriores, habría observabilidad parcial; ampliaríamos las ventanas temporales o consideraríamos una red recurrente. Tener 256 entradas, por sí solo, no garantiza que DQN aprenda una buena política.


### 1.1.b 

El conjunto $\mathcal{A}=\{\text{ignorar},\text{alertar},\text{bloquear},\text{aislar}\}$ **encaja directamente con DQN y sus variantes**. Con cuatro salidas, la red calcula todos los valores en una sola evaluación y la acción codiciosa se obtiene mediante

$$
a_t=\arg\max_{a\in\mathcal A}Q(s_t,a;\mathbf w).
$$

Comparar cuatro números es barato, tanto para actuar como para calcular el máximo del target. No hace falta otro algoritmo por la dimensionalidad continua del estado.

Usaríamos DQN como referencia experimental y consideraríamos **Double DQN con Prioritized Experience Replay (PER)**: Double DQN reduce el sesgo de sobreestimación y PER mejora el aprovechamiento de transiciones con alto error TD. Son modificaciones compatibles que atienden problemas distintos. Dueling también sería compatible, aunque las cuatro acciones no lo hacen obligatorio.

Esta conclusión supone que cada acción está bien definida, por ejemplo, que la duración del bloqueo y el segmento que se aísla están determinados por el entorno. Si el agente también tuviera que elegir una duración continua, cambiaría el espacio de acciones y habría que reconsiderar la formulación. En el problema planteado no se necesita ese cambio.

### 1.1.c 

Sea $p<0.001$ la probabilidad de que una transición sea crítica, $N$ el número de transiciones de un buffer lleno y $K$ la cantidad de transiciones críticas almacenadas. Si el buffer representa un proceso con esa frecuencia estable,

$$
\mathbb E[K]=Np<0.001N.
$$

Por ejemplo, con $N=10\,000$ habría **menos de 10 transiciones críticas en promedio**. Bajo el supuesto adicional de eventos independientes, $K\sim\operatorname{Binomial}(N,p)$ y

$$
P(K=0)=(1-p)^N.
$$

Con $N=1\,000$ y $p=0.001$ como referencia límite, esta probabilidad es aproximadamente $36.8\%$; con una frecuencia menor aumenta. Las intrusiones reales pueden agruparse temporalmente, por lo que el modelo binomial es una simplificación, no una propiedad garantizada del tráfico. Además, el reemplazo FIFO puede eliminar una experiencia crítica antes de que se aproveche suficientemente.

**Muestreo uniforme.** Cada transición almacenada tiene probabilidad $1/N$, por lo que la probabilidad de obtener una crítica en una extracción es $K/N$, no $1/2$. Para un minibatch de tamaño $B$, la cantidad de ejemplos críticos $X$, condicionada al buffer, satisface

$$
\mathbb E[X\mid K]=B\frac{K}{N}.
$$

Con reemplazo, $P(X=0\mid K)=(1-K/N)^B$; sin reemplazo,

$$
P(X=0\mid K)=\frac{\binom{N-K}{B}}{\binom NB}.
$$

Cuando $K/N\approx p$, usando $B=64$ y $p=0.001$ como referencia, se obtiene $\mathbb E[X]\approx0.064$ y $P(X=0)\approx0.999^{64}\approx93.8\%$. Por tanto, casi todos los minibatches carecerían de ejemplos críticos. El replay uniforme reduce la correlación temporal, pero **no equilibra las clases**: la mayoría de actualizaciones usa tráfico normal. Esto dificulta aprender las consecuencias de decisiones durante ataques, aunque el tamaño del gradiente también depende de las recompensas y del error TD.

**Variante propuesta: Prioritized Experience Replay.** Para cada transición $i$, definimos

$$
\delta_i=y_i-Q(s_i,a_i;\mathbf w),\qquad
p_i=|\delta_i|+\epsilon,\qquad
P(i)=\frac{p_i^\alpha}{\sum_{j=1}^{N}p_j^\alpha},
$$

donde $\epsilon>0$ evita probabilidades nulas y $\alpha\in[0,1]$ regula la priorización; $\alpha=0$ recupera el muestreo uniforme. Una intrusión mal predicha, especialmente con consecuencias de gran magnitud, puede generar un error TD alto y reutilizarse con mayor frecuencia. Las prioridades se actualizan al recalcular los errores.

El muestreo deja de ser uniforme. Para corregir su sesgo respecto al objetivo de replay uniforme, pueden ponderarse las pérdidas con pesos de importancia

$$
\omega_i=(N P(i))^{-\beta},\qquad 0\leq\beta\leq1,
$$

habitualmente normalizados por un máximo común. Con $\beta=1$, el factor sin normalizar satisface $\mathbb E_{i\sim P}[\omega_i\ell_i]=\frac1N\sum_i\ell_i$ para una pérdida por transición $\ell_i$; valores menores aplican una corrección parcial. Se prioriza dónde aprender sin confundir esa frecuencia de entrenamiento con la prevalencia real de ataques.

**Alcance:** PER atiende la ineficiencia del muestreo, pero prioriza sorpresa, no rareza ni importancia de seguridad directamente. No garantiza que todo ataque tenga alto error TD, no genera ataques ausentes y no impide su expulsión FIFO. Si faltara cobertura, lo complementaríamos con retención de incidentes o muestreo estratificado, ajustando los pesos a la distribución efectivamente utilizada. Double DQN por sí solo no soluciona este problema de muestreo.


### 1.1.d 

Si solamente penalizáramos los falsos positivos, por ejemplo $r_{t+1}=-\mathbf1\{\text{falso positivo}\}$, el agente podría aprender a **ignorar todo el tráfico**. Así no emitiría falsas alarmas ni bloquearía usuarios legítimos, pero dejaría pasar todas las intrusiones. Reducir el 8% de falsos positivos a 0% no demostraría una mejora de seguridad: esa tasa se calcula sobre tráfico benigno y no mide los falsos negativos.

Proponemos recompensar la contención efectiva y penalizar tanto el daño de ataques como el de intervenciones innecesarias. Definimos, para cada transición:

- $z_t\in\{0,1\}$: indica si hay una intrusión real, verificada por el entorno de entrenamiento o por análisis posterior.
- $c_t\in\{0,1\}$: indica que en esa transición se contiene exitosamente la intrusión; se recompensa una sola vez por incidente.
- $h_t\in[0,1]$: daño de seguridad durante el intervalo, medido en una escala fija; $h_t=1$ representa una unidad de daño grave por una intrusión no contenida.
- $f(a_t)$: costo adicional de una intervención sobre tráfico benigno.
- $o(a_t)$: costo operativo de ejecutar una acción, haya o no intrusión.

Una recompensa ilustrativa de **cuatro componentes** sería

$$
\boxed{r_{t+1}=20z_tc_t-1000z_th_t-(1-z_t)f(a_t)-o(a_t).}
$$

| Acción | $f(a)$: intervención injustificada | $o(a)$: costo operativo |
|---|---:|---:|
| Ignorar | 0 | 0 |
| Alertar | 1 | 0.1 |
| Bloquear temporalmente | 10 | 0.5 |
| Aislar el segmento | 50 | 2 |

La magnitud relativa responde a estos objetivos:

1. **Daño por intrusión ($-1000z_th_t$):** es el costo dominante de un evento grave, porque un compromiso puede causar pérdidas mucho mayores que una alerta innecesaria. Penaliza las consecuencias de omitir o retrasar una respuesta efectiva.
2. **Contención verificada ($+20z_tc_t$):** reconoce resolver un ataque, sin premiar simplemente emitir alertas o repetir bloqueos. Es menor que la penalización por daño grave para que detectar tarde no compense arbitrariamente el daño producido.
3. **Intervención injustificada ($-(1-z_t)f(a_t)$):** aislar un segmento legítimo cuesta más que bloquear una conexión, y bloquear cuesta más que alertar. Este término desalienta resolver la seguridad interrumpiendo indiscriminadamente el servicio.
4. **Costo operativo ($-o(a_t)$):** representa recursos de análisis y ejecución. Entre dos acciones igualmente efectivas, favorece la menos costosa; es menor que el costo de dañar tráfico legítimo y mucho menor que el de una intrusión grave.

Por ejemplo, ignorar tráfico benigno produce $0$; alertar sobre él, $-1.1$; bloquearlo, $-10.5$; aislarlo, $-52$. Contener por primera vez un ataque mediante bloqueo y sin daño en ese intervalo produce $19.5$, mientras que ignorarlo y permitir $h_t=1$ produce $-1000$. Si alertar activa una respuesta humana que contiene el ataque después, ese beneficio aparece en las transiciones posteriores y contribuye al retorno descontado.

Los valores son una **propuesta de escala, no costos estimados de la empresa**. Deben calibrarse con el daño real, la duración del paso temporal y la prevalencia de ataques. En una comparación simplificada de un paso, si $q=P(z_t=1\mid s_t)$, ignorar un ataque siempre genera $h_t=1$ y bloquear siempre lo contiene sin daño, entonces

$$
\mathbb E[r\mid\text{ignorar}]=-1000q,\qquad
\mathbb E[r\mid\text{bloquear}]=19.5q-10.5(1-q)=30q-10.5.
$$

Bloquear sería preferible si $q>10.5/1030\approx1.02\%$. Esto muestra cómo los pesos fijan un intercambio entre seguridad y disponibilidad: con ataques muy raros hace falta distinguir estados de riesgo elevado; los coeficientes no garantizan por sí solos una política útil. En el MDP completo también cuentan los efectos futuros, mediante $G_t=\sum_{k=0}^{\infty}\gamma^k r_{t+k+1}$.

Evaluaríamos la política frente al sistema de reglas usando conjuntamente falsos positivos, ataques omitidos, daño y disponibilidad. La verdad de referencia que determina $z_t$, $c_t$ y $h_t$ debe provenir de resultados verificados, no de las propias predicciones del agente, para evitar recompensarlo por declarar que acertó.


## Task 1.2

El equipo propone usar Double DQN en lugar de DQN estándar.

a. Expliquen formalmente el problema de sobreestimación que Double DQN resuelve. En el contexto
específico de detección de intrusiones, ¿qué consecencia operacional tendría que el agente
sobreestime sistemáticamente el valor de la acción "bloquear"? ¿Y de la acción "ignorar"?

b. Escriban la expresión del target de DQN estándar y la del target de Double DQN, identificando con
precisión qué parámetros hacen qué en cada caso. ¿Cuál es el único cambio de implementación
entre ambos?

c. Argumenten si en este dominio específico la sobreestimación es un problema más grave en ciertos
tipos de estados que en otros. Consideren estados con alta ambigüedad (tráfico borderline) versus
estados claramente maliciosos.

### 1.2.a 

En DQN, el máximo del target selecciona y evalúa una acción utilizando las mismas estimaciones de la red objetivo. Para formalizar el sesgo, fijemos un siguiente estado $s'$ y supongamos

$$
Q(s',a;\mathbf w^-)=q_*(s',a)+\varepsilon_a,
\qquad \mathbb E[\varepsilon_a]=0.
$$

Aunque cada estimación sea individualmente insesgada, la convexidad del máximo implica

$$
\mathbb E\!\left[\max_a\big(q_*(s',a)+\varepsilon_a\big)\right]
\geq
\max_a\mathbb E\!\left[q_*(s',a)+\varepsilon_a\right]
=\max_a q_*(s',a).
$$

El máximo tiende a escoger acciones cuyo error fue favorable. La desigualdad puede ser estricta cuando el ruido cambia la acción ganadora; no significa que cada estimación esté siempre sobreestimada. Por ejemplo, con dos acciones de valor real cero y errores independientes $+1$ o $-1$ equiprobables, el máximo vale $+1$ con probabilidad $3/4$ y $-1$ con probabilidad $1/4$: su esperanza es $0.5$, aunque el mejor valor real es $0$. Ese optimismo entra en el target multiplicado por $\gamma$ y puede propagarse mediante bootstrapping.

**Double DQN reduce este efecto separando selección y evaluación:** la red principal elige la acción y la red objetivo estima su valor. Ya no se toma directamente el mayor error de la red que evalúa. No elimina todo sesgo: ambas redes están relacionadas por las copias periódicas de parámetros, por lo que sus errores pueden seguir correlacionados; incluso puede haber subestimación.

En detección de intrusiones, si la sobreestimación altera la preferencia entre acciones:

- **Sobreestimar «bloquear»** puede llevar a bloquear tráfico legítimo innecesariamente, aumentar falsos positivos e interrumpir servicios. El agente interpreta que bloquear tiene más beneficio o menos costo del real.
- **Sobreestimar «ignorar»** puede llevar a omitir respuestas ante ataques, aumentar falsos negativos, retrasar la contención y permitir propagación o exfiltración. Que el valor real sea negativo no impide la sobreestimación: pasar de $-100$ a $-5$ ya hace parecer mucho mejor esa acción.

Un desplazamiento idéntico de todos los valores no cambia necesariamente la decisión inmediata; el riesgo operacional surge especialmente cuando el error cambia cuál acción parece mejor.


### 1.2.b 

Sea $(s,a,r,s',d)$ una transición, con $r=r_{t+1}$ y $d=1$ si $s'$ es un estado terminal real; en otro caso $d=0$. Denotamos por $\mathbf w$ los parámetros de la **red principal**, actualizados por gradiente, y por $\mathbf w^-$ los de la **red objetivo**, mantenidos fijos entre copias periódicas $\mathbf w^-\leftarrow\mathbf w$. Ambas redes tienen la misma arquitectura.

**DQN estándar:**

$$
\boxed{y^{\mathrm{DQN}}=r+\gamma(1-d)\max_{a'\in\mathcal A}Q(s',a';\mathbf w^-).}
$$

La red objetivo $\mathbf w^-$ realiza ambas funciones: el máximo identifica la acción y devuelve su valor usando esa misma red.

**Double DQN:**

$$
a_{\mathrm{sel}}=\arg\max_{a'\in\mathcal A}Q(s',a';\mathbf w),
$$
$$
\boxed{y^{\mathrm{DDQN}}=r+\gamma(1-d)Q(s',a_{\mathrm{sel}};\mathbf w^-).}
$$

Aquí $\mathbf w$ **selecciona** la acción y $\mathbf w^-$ la **evalúa**. El factor $(1-d)$ evita bootstrap al terminar un episodio: en ambos métodos, si $d=1$, el target es $r$. Un corte artificial por límite de tiempo no necesariamente representa un estado terminal del MDP.

**El único cambio algorítmico entre las versiones base está en el cálculo del valor del siguiente estado dentro del target:** sustituir el máximo de la red objetivo por el valor que esta asigna a la acción seleccionada por la red principal. En pseudocódigo para un minibatch:

```python
# Todo este bloque se calcula sin construir gradientes.
q_target = target_network(next_states)       # Forma: [B, 4]

# DQN:
next_value_dqn = max_over_actions(q_target)  # Forma: [B]

# Double DQN, en sustitución de la línea anterior:
q_online = online_network(next_states)      # Forma: [B, 4]
selected = argmax_over_actions(q_online)    # Forma: [B]
next_value_ddqn = q_target[rows, selected]   # rows = 0, ..., B-1

target = rewards + gamma * (1 - terminal) * next_value_ddqn
```

Se conservan la arquitectura, el buffer, la política de exploración y el mecanismo de actualización de la red objetivo. DQN ya tenía dos redes; Double DQN no requiere agregar una tercera. Sí puede requerir una evaluación adicional de la red principal en $s'$, según qué cálculos se reutilicen.

En ambos casos se minimiza, por ejemplo,

$$
L(\mathbf w)=\frac1B\sum_{i=1}^{B}
\left(\operatorname{stopgrad}(y_i)-Q(s_i,a_i;\mathbf w)\right)^2.
$$

El gradiente actualiza $\mathbf w$ a través de $Q(s_i,a_i;\mathbf w)$, **no a través del target**, ni siquiera cuando $\mathbf w$ interviene en la selección de Double DQN. Si además se usa PER, se ponderan los términos con $\omega_i$; eso es una modificación independiente.


### 1.2.c 

La sobreestimación **no tiene el mismo efecto en todos los estados**. Conviene distinguir la probabilidad de elegir una acción equivocada de la gravedad de sus consecuencias.

En tráfico ambiguo, como un aumento de conexiones que podría ser una copia de respaldo o un ataque, suele haber incertidumbre y valores relativamente cercanos para ignorar, alertar y bloquear. Sea $a^*$ la mejor acción real y definamos la separación respecto a otra acción $b$:

$$
\Delta_b(s)=q_*(s,a^*)-q_*(s,b)>0.
$$

Si $\hat Q(s,a)=q_*(s,a)+\varepsilon_a$, la acción inferior $b$ supera a $a^*$ cuando

$$
\varepsilon_b-\varepsilon_{a^*}>\Delta_b(s).
$$

Para una misma distribución de errores, una separación pequeña hace más fácil invertir el orden. En estados ambiguos, el máximo puede seleccionar con frecuencia una estimación demasiado optimista; esto puede causar bloqueos innecesarios o ignorar ataques incipientes. Double DQN es especialmente útil para reducir el optimismo que se introduce en los targets en estas situaciones, aunque no resuelve la ambigüedad de las observaciones.

En estados claramente maliciosos, **si están bien representados en los datos y la recompensa está bien diseñada**, las acciones efectivas de contención deberían tener valores claramente superiores a ignorar. Si la separación supera ampliamente el error de estimación, un error moderado no cambiaría la acción elegida. En ese sentido, la decisión sería más robusta.

Sin embargo, «claramente malicioso» para un analista no implica «bien aprendido» por la red. Dado que las intrusiones son raras, esos estados pueden tener poco soporte en el replay y errores grandes. Además, confundir bloquear con ignorar durante un ataque grave puede ser mucho más costoso que una decisión incorrecta en tráfico ambiguo. Incluso ante un ataque evidente, bloquear y aislar podrían tener valores cercanos entre sí.

Por tanto, esperaríamos mayor sensibilidad al ruido en estados ambiguos con acciones de valor similar, pero potencialmente mayor daño por error en ataques graves poco representados. La evaluación debe separar ambos tipos de estados. **Double DQN aborda el sesgo del máximo y PER ayuda a reutilizar experiencias mal aprendidas; ninguno garantiza seguridad ni buena cobertura por sí solo.**


## Referencias 

- **[P]** [Presentación *S12 - Deep RL P1*](../S12%20-%20Deep%20RL%20P1.pdf), *Deep RL Parte 1: DQN y Variantes*, 2026. Experience Replay (4), Target Network (5), Double DQN (8), PER (10) y limitaciones (12).
- **[SB]** Richard S. Sutton y Andrew G. Barto, [*Reinforcement Learning: An Introduction*] **“Second edition, © 2014, 2015**.
